# Sandbox для интерактивного просмотра SAE-нейронов

Этот ноутбук НЕ запускает обучение и НЕ извлекает активации.
Он только грузит готовые артефакты из `./artifacts/` и позволяет
интерактивно полистать нейроны.

Перед запуском убедитесь, что у вас есть:
- `artifacts/activations.npz` — от `scripts/01_extract.py`
- `artifacts/samples.json` — там же
- `artifacts/sae_<mode>.pt` — от `scripts/02_train_sae.py`
- `artifacts/features_<mode>.npz`, `neuron_stats_<mode>.npz` — от `scripts/03_analyze.py`

In [ ]:
# Делаем src/ импортируемым, если ноутбук открыт из корня проекта.
import sys, os
from pathlib import Path
PROJECT_ROOT = Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

import numpy as np
import torch

from src.io_utils import load_activations, load_config, load_sae
from src.data import load_saved_samples
from src.analysis import (
    encode_all, neuron_statistics, select_top_neurons,
    top_contexts_for_neuron, label_contrast, sample_level_contrast,
    find_contrast_neurons,
)

In [ ]:
# Конфиг и выбор SAE.
cfg = load_config('configs/default.yaml')
MODE = cfg['sae']['mode']   # или вручную: 'topk', 'jumprelu', 'relu_l1'

ART = Path(cfg['paths']['artifacts_dir'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'

pack = load_activations(ART / 'activations.npz')
samples = load_saved_samples(ART / 'samples.json')
sae, ckpt = load_sae(ART / f'sae_{MODE}.pt', device=device)
print(f'mode={MODE}, d_hidden={sae.d_hidden}, samples={len(samples)}')

In [ ]:
# Features и статистика — если есть кеш на диске, загружаем; иначе считаем.
feat_path = ART / f'features_{MODE}.npz'
stats_path = ART / f'neuron_stats_{MODE}.npz'

if feat_path.exists():
    features = np.load(feat_path)['features']
else:
    features = encode_all(sae, pack['hidden'], batch_size=cfg['analysis']['encode_batch_size'], device=device)

if stats_path.exists():
    stats = {k: np.load(stats_path)[k] for k in np.load(stats_path).files}
else:
    stats = neuron_statistics(features)

print(f'features.shape = {features.shape}')
print(f"живых нейронов: {(stats['fire_rate'] > 0).sum()}")

In [ ]:
# Загрузим токенайзер — нужен для top_contexts_for_neuron (декодирование сниппетов).
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg['model']['name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Интерактивный просмотр нейрона

Меняйте `NEURON_ID` и перезапускайте ячейку ниже.

In [ ]:
NEURON_ID = 16585   # подставьте свой

print(f"=== Нейрон #{NEURON_ID} ===")
print(f"  fire_rate   = {stats['fire_rate'][NEURON_ID]:.4f}")
print(f"  mean_active = {stats['mean_active'][NEURON_ID]:.3f}")
print(f"  n_fires     = {stats['n_fires'][NEURON_ID]}")
print()
for c in top_contexts_for_neuron(
        NEURON_ID, features, pack['sample_idx'], pack['token_pos'],
        samples, tokenizer, top_k=15, window=30):
    sn = c['snippet'].replace('\n', ' ')[:140]
    # token_pos выведен дополнительно — см. HANDOVER.md, проблема #1
    print(f"  [act={c['activation']:5.2f}, {c['label']:>3}, file{c['file_idx']}, pos={c['token_pos']:4d}] ...{sn}")

## Сравнение decoder-векторов нескольких нейронов

Если у нескольких нейронов один и тот же топ-контекст, имеет смысл
проверить их decoder-веса: близкие по косинусу → feature splitting,
ортогональные → разные направления, активирующиеся на одних примерах.

In [ ]:
candidates = [16585, 11787, 18580, 14471, 16777]  # подставьте свои
W_dec_np = sae.W_dec.detach().cpu().numpy()       # (d_hidden, d_in)

print('Cosine similarity:')
print(f"{'':>10}" + ''.join(f'{n:>8}' for n in candidates))
for n1 in candidates:
    v1 = W_dec_np[n1]; v1 = v1 / (np.linalg.norm(v1) + 1e-8)
    print(f'{n1:>10}', end='')
    for n2 in candidates:
        v2 = W_dec_np[n2]; v2 = v2 / (np.linalg.norm(v2) + 1e-8)
        print(f'{(v1*v2).sum():>8.3f}', end='')
    print()

## Контраст Yes vs No: топовые триггеры


In [ ]:
tok_c = label_contrast(features, pack['label'],
                       pos_label=cfg['contrast']['pos_label'],
                       neg_label=cfg['contrast']['neg_label'])
smp_c = sample_level_contrast(features, pack['sample_idx'], samples,
                              pos_label=cfg['contrast']['pos_label'],
                              neg_label=cfg['contrast']['neg_label'])

yes_neurons, no_neurons = find_contrast_neurons(
    tok_c, smp_c,
    top_k=cfg['contrast']['top_k'],
    pool_factor=cfg['contrast']['pool_factor'],
)
print('Yes-trigger neurons:', yes_neurons)
print('No-trigger neurons: ', no_neurons)

In [ ]:
# Покажем top контексты по одному из триггеров
for nid in yes_neurons[:3]:
    print(f'\n=== YES-trigger #{nid}  d_tok={tok_c["cohens_d"][nid]:+.2f}  d_smp={smp_c["cohens_d"][nid]:+.2f} ===')
    for c in top_contexts_for_neuron(
            nid, features, pack['sample_idx'], pack['token_pos'],
            samples, tokenizer, top_k=5, window=25):
        sn = c['snippet'].replace('\n', ' ')[:130]
        print(f"  [act={c['activation']:5.2f}, {c['label']:>3}, file{c['file_idx']}] ...{sn}")